In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [ ]:
# Task 1:
golden_path = os.path.join(path, 'Q3_data.csv')
df_golden = pd.read_csv(golden_path)


In [ ]:
# Task 2:
print(f"Dataset shape: {df_golden.shape}")
df_golden.head()

In [ ]:
# Task 3:
df_golden.info()

In [ ]:
df_golden.count()


#the data is huge so i assue any value >20001 has a missing. and solve the entire Q with this problem :( i ask 2 TAs

In [ ]:
# Task 4:
df_golden.describe()

In [ ]:
# Task 1:
df_clean = df_golden[df_golden.count()].copy()

df_clean['P_2'] = df_clean['P_2'].fillna(df_clean['P_2'].mean())
df_clean['B_2'] = df_clean['B_2'].fillna(df_clean['B_2'].mean())
df_clean['D_142'] = df_clean['D_142'].fillna(df_clean['D_142'].mean())
df_clean['D_143'] = df_clean['D_143'].fillna(df_clean['D_143'].mean())
df_clean['D_144'] = df_clean['D_144'].fillna(df_clean['D_144'].mean())
df_clean['D_145'] = df_clean['D_145'].fillna(df_clean['D_145'].mean())

print("Missing values remaining:", df_clean.isnull().sum().sum())



In [ ]:
# Task 2:
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(train_df)

In [ ]:
# Task 3:
categorical_cols = [df_golden.count()]
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 4:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)


In [ ]:
# Task 5:
print(f"\nFeature ranges - Min: {X_train.min().min():.2f}, Max: {X_train.max().max():.2f}")
X_train.head(3)

In [ ]:
# Task 1:
feature_cols = ['P_2', 'D_39', 'B_1', 'B_2','R_1']

X = df_clean[feature_cols]
y = df_clean['R_1']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5:
model = CatBoostClassifier(
      verbose=0,
      n_estimators=200,
      max_depth=4
  )

model.fit(X_train_scaled, y_train)
print("Model trained!")

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

sr_results['acc'].append(accuracy)
sr_results['f1'].append(f1)

print(f"  Accuracy:  {np.mean(sr_results['acc']):.4f}")
print(f"  F1-Score:  {np.mean(sr_results['f1']):.4f}")

#kfold = KFold(n_splits=5, shuffle=True, random_state=42)

#for train_idx, val_idx in kfold.split(X_train_scaled):
#    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
#    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
   # model.fit(X_fold_train, y_fold_train)
   # y_fold_pred = model.predict(X_fold_val)

#print(f"5-Fold CV Results:")

n_splits = 3 # K=3 Folds

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

sr_results = {'loss': [], 'acc': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # Get the train & test split for this fold
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train using gradient descent with learning rate = 0.5
  theta, losses = gradient_descent(X_train, y_train, lr=0.5, num_classes=4)

  # Calculate z & class probabilities for X_test
  z = np.dot(X_test, theta)
  y_pred_proba = softmax(z)

  # Pick the predicted classes with the highest probability
  y_pred = np.argmax(y_pred_proba, axis=1)

  # Calculate evaluation metrics
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, average='macro')  # for multiclass f1 score, you should set the average hyperparameter ("macro", "micro", "weighted")

  # Store results
  sr_results['loss'].append(losses)
  sr_results['acc'].append(accuracy)
  sr_results['f1'].append(f1)

  avg_loss = np.mean(sr_results['loss'], axis=0)



In [ ]:
# Task 1:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2:
plt.figure(figsize=(10, 5))
plt.hist(df_golden['R_1'].dropna(), bins=30, edgecolor='black', color='orange')
plt.title('R_1 Distribution')
plt.xlabel('R_1')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: